# Exercise 1B — Advanced Imaging Design Challenge — Sample Solution

In [ ]:
from pathlib import Path
import sys, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().parents[1] if Path.cwd().name == "solutions" else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(repo_root / "src"))

from skimage import io, img_as_float, color
import cv2

from cvis_utils import (
    load_gray, show_images,
    sobel_energy, contrast_between_regions, cnr_between_regions, weighted_gray,
    max_exposure_time_for_blur, required_pixels_for_feature, required_fov_for_pixel_count,
    robust_zscore_map, anomaly_score_from_map, iou
)

DATA = repo_root / "data"
ADV = DATA / "advanced"

## Part D — Quantitative imaging design

In [ ]:
feature_size_mm = 0.5
n_px = 2448
target_fov_mm = 400.0
desired_px = np.array([2, 3, 5, 10])

rows = []
for p in desired_px:
    required_mm = required_pixels_for_feature(feature_size_mm, p)
    max_fov = required_fov_for_pixel_count(required_mm, n_px)
    feasible = max_fov >= target_fov_mm
    rows.append({
        "desired_px_per_feature": int(p),
        "required_mm_per_px": required_mm,
        "max_fov_mm_with_2448_px": max_fov,
        "feasible_for_400mm_fov": feasible
    })
df_design = pd.DataFrame(rows)
df_design

In [ ]:
object_speed_mm_s = 200.0
mm_px_current = target_fov_mm / n_px
blur_tolerances_px = np.array([1, 2, 4])

rows = []
for b in blur_tolerances_px:
    t_s = max_exposure_time_for_blur(b, mm_px_current, object_speed_mm_s)
    rows.append({
        "blur_tolerance_px": int(b),
        "max_exposure_s": t_s,
        "max_exposure_ms": 1000 * t_s
    })
df_exposure = pd.DataFrame(rows)
df_exposure

In [ ]:
answer_D = """
For a 0.5 mm feature, 2 pixels allows a large field of view but is only a theoretical lower bound.
At 5 pixels per feature, a 2448 px camera supports about 245 mm FoV, so a 400 mm FoV is not sufficient for robust localization.
At the current 400 mm FoV, a 1 px blur tolerance permits only about 0.82 ms exposure at 200 mm/s.
The first levers I would consider are reducing FoV, increasing sensor/lens resolution, stronger illumination for shorter exposure, or reducing process speed / using strobe lighting.
"""
print(answer_D)

## Part E — Spectral and channel contrast

In [ ]:
rgb = img_as_float(io.imread(ADV / "spectral" / "material_rgb.png"))
nir = load_gray(ADV / "spectral" / "material_nir_like.png")
uv = load_gray(ADV / "spectral" / "material_uv_like.png")
defect_mask = load_gray(ADV / "spectral" / "material_defect_mask.png") > 0.5
gray = color.rgb2gray(rgb)
background_mask = ~defect_mask

channels = {
    "R": rgb[..., 0],
    "G": rgb[..., 1],
    "B": rgb[..., 2],
    "gray": gray,
    "NIR-like": nir,
    "UV-like": uv,
}

rows = []
for name, img in channels.items():
    rows.append({
        "channel": name,
        "contrast": contrast_between_regions(img, defect_mask, background_mask),
        "CNR": cnr_between_regions(img, defect_mask, background_mask)
    })
df_contrast = pd.DataFrame(rows).sort_values("CNR", ascending=False)
df_contrast

In [ ]:
weight_candidates = [
    (1.0, 0.0, 0.0),
    (0.0, 1.0, 0.0),
    (0.0, 0.0, 1.0),
    (0.8, 0.1, 0.1),
    (0.6, 0.2, 0.2),
    (0.5, 0.4, 0.1),
    (0.3, 0.6, 0.1),
]

rows = []
best_img = None
best_weights = None
best_cnr = -np.inf

for w in weight_candidates:
    img_w = weighted_gray(rgb, w)
    cnr = cnr_between_regions(img_w, defect_mask, background_mask)
    rows.append({"weights_RGB": w, "CNR": cnr, "contrast": contrast_between_regions(img_w, defect_mask, background_mask)})
    if cnr > best_cnr:
        best_cnr = cnr
        best_weights = w
        best_img = img_w

df_weights = pd.DataFrame(rows).sort_values("CNR", ascending=False)
df_weights

In [ ]:
show_images(
    [rgb, gray, best_img, nir, defect_mask],
    ["RGB", "standard gray", f"best weighted RGB {best_weights}", "NIR-like", "defect mask"],
    ncols=5,
    figsize=(15, 4),
)

In [ ]:
answer_E = """
The synthetic NIR-like image gives the strongest defect CNR because the defect was designed to have a stronger response outside standard RGB.
A weighted RGB image can improve over standard grayscale, but it still may not match a spectral band that creates materially stronger contrast.
The industrial conclusion is that monochrome/color/NIR choices should be task-driven and validated on the material, not selected by default.
"""
print(answer_E)

## Part F — Line scan and temporal sampling

In [ ]:
web = load_gray(ADV / "line_scan" / "continuous_web_texture.png")
line_med = load_gray(ADV / "line_scan" / "line_scan_medium_rate.png")
line_under = load_gray(ADV / "line_scan" / "line_scan_under_sampled.png")

show_images([web, line_med, line_under], ["original continuous texture", "medium line rate", "under-sampled line scan"], ncols=3, figsize=(14,4))

In [ ]:
web_speed_m_s = 1.5
delta_y_req_mm = 0.10

web_speed_mm_s = web_speed_m_s * 1000.0
f_line_required = web_speed_mm_s / delta_y_req_mm

print(f"Required line rate: {f_line_required:.0f} lines/s")

In [ ]:
row = web.shape[0] // 2

plt.figure(figsize=(12, 4))
plt.plot(web[row, :], label="original", alpha=0.8)
plt.plot(line_med[row, :], label="medium line rate", alpha=0.8)
plt.plot(line_under[row, :], label="under-sampled", alpha=0.8)
plt.legend()
plt.grid(True, alpha=0.3)
plt.title("Line profiles across web texture")
plt.xlabel("column")
plt.ylabel("intensity")
plt.show()

print("Sobel energy original:", sobel_energy(web))
print("Sobel energy medium  :", sobel_energy(line_med))
print("Sobel energy under   :", sobel_energy(line_under))

In [ ]:
answer_F = """
The under-sampled line scan loses fine spatial structure and can produce misleading repeated patterns.
In a real line-scan system, an encoder helps synchronize line acquisition with actual web motion, especially when conveyor speed fluctuates.
Without synchronization, spatial sampling in the motion direction can vary and distort measurements.
"""
print(answer_F)

## Part G — Homography-based rectification and measurement

In [ ]:
warped = img_as_float(io.imread(ADV / "homography" / "planar_target_warped.png"))
rectified_reference = img_as_float(io.imread(ADV / "homography" / "planar_target_rectified.png"))

with open(ADV / "homography" / "homography_metadata.json", "r") as f:
    hmeta = json.load(f)

src_warped = np.array(list(hmeta["fiducials_warped_px"].values()), dtype=np.float32)
dst_rectified = np.array(list(hmeta["fiducials_rectified_px"].values()), dtype=np.float32)
output_w, output_h = hmeta["rectified_size_px"]

H, status = cv2.findHomography(src_warped, dst_rectified)
rectified = cv2.warpPerspective((warped*255).astype(np.uint8), H, (output_w, output_h))
rectified = rectified.astype(float) / 255.0

show_images([warped, rectified, rectified_reference], ["warped", "rectified by solution", "reference"], ncols=3, figsize=(14, 5))
print("H =")
print(H)

In [ ]:
mm_per_px_rectified = hmeta["assumed_mm_per_px_rectified"]
x1, y1, x2, y2 = hmeta["measurement_object_rectified_px"]

width_px = x2 - x1
height_px = y2 - y1
width_mm = width_px * mm_per_px_rectified
height_mm = height_px * mm_per_px_rectified

print(f"Object width:  {width_px:.1f} px = {width_mm:.2f} mm")
print(f"Object height: {height_px:.1f} px = {height_mm:.2f} mm")

In [ ]:
answer_G = """
Measurement on the warped image is risky because perspective changes local scale across the image.
The homography corrects a planar projective distortion and maps the inspection plane into a rectified coordinate system.
It does not correct non-planarity, lens distortion that was not modeled, blur, poor calibration target detection, or errors in the assumed physical scale.
"""
print(answer_G)

## Part H — Simple non-ML anomaly baseline

In [ ]:
train_dir = ADV / "anomaly_baseline" / "train_nominal"
test_dir = ADV / "anomaly_baseline" / "test"
mask_dir = ADV / "anomaly_baseline" / "masks"
manifest = pd.read_csv(ADV / "anomaly_baseline" / "manifest.csv")

train_imgs = [load_gray(p) for p in sorted(train_dir.glob("*.png"))]
train_stack = np.stack(train_imgs, axis=0)

median_img = np.median(train_stack, axis=0)
mad_img = np.median(np.abs(train_stack - median_img), axis=0)

show_images([median_img, mad_img], ["nominal median image", "nominal MAD image"], ncols=2, figsize=(10, 4))

In [ ]:
rows = []
score_maps = {}

for _, row in manifest.iterrows():
    img = load_gray(test_dir / row["image"])
    smap = robust_zscore_map(img, median_img, mad_img, eps=1e-3)
    score = anomaly_score_from_map(smap, percentile=99)
    score_maps[row["image"]] = smap
    rows.append({
        "image": row["image"],
        "label": row["label"],
        "is_anomaly": int(row["is_anomaly"]),
        "score": score
    })

df_scores = pd.DataFrame(rows).sort_values("score", ascending=False)
df_scores

In [ ]:
nominal_scores = df_scores.loc[df_scores["is_anomaly"] == 0, "score"]
threshold = nominal_scores.mean() + 3 * nominal_scores.std()

df_scores["predicted_anomaly"] = df_scores["score"] > threshold
tp = int(((df_scores["predicted_anomaly"] == 1) & (df_scores["is_anomaly"] == 1)).sum())
fp = int(((df_scores["predicted_anomaly"] == 1) & (df_scores["is_anomaly"] == 0)).sum())
fn = int(((df_scores["predicted_anomaly"] == 0) & (df_scores["is_anomaly"] == 1)).sum())
tn = int(((df_scores["predicted_anomaly"] == 0) & (df_scores["is_anomaly"] == 0)).sum())

print("threshold:", threshold)
print({"TP": tp, "FP": fp, "FN": fn, "TN": tn})
df_scores.sort_values("score", ascending=False)

In [ ]:
top = df_scores.sort_values("score", ascending=False).iloc[0]
nominal_near = df_scores[df_scores["is_anomaly"] == 0].assign(dist=lambda d: np.abs(d["score"] - threshold)).sort_values("dist").iloc[0]

imgs = []
titles = []
for row in [top, nominal_near]:
    img = load_gray(test_dir / row["image"])
    smap = score_maps[row["image"]]
    imgs += [img, smap]
    titles += [f'{row["image"]}\nscore={row["score"]:.2f}', "score map"]

show_images(imgs, titles, ncols=2, figsize=(10, 8), cmap="gray")

In [ ]:
advanced_report = """
1. Design calculation:
The most important calculation was pixels per defect / mm-per-pixel, because it determines whether the visual evidence is resolvable at all.

2. Spectral/channel choice:
The NIR-like channel produced much stronger defect contrast in this synthetic example, showing that spectral choice can dominate model choice.

3. Line-scan / temporal sampling:
Under-sampling in the scan direction can distort texture and create misleading visual structure; encoder synchronization is important when speed varies.

4. Homography and measurement:
Homography rectifies a planar inspection surface, making object-space measurement more meaningful, but only under the planar-scene assumption.

5. Simple anomaly baseline:
The pixelwise robust z-score baseline can detect strong deviations, but it is fragile to illumination change, alignment error, non-stationary texture, and structured normal variation.
"""
print(advanced_report)